# Agent Trigger (Phase 1)

This notebook implements a **single Trigger agent** with local in-memory simulation only.

Phase 1 scope:
- Load configuration with `simulated_city.config.load_config()`
- Create person states with deterministic randomness
- Run a local movement loop with edge reflection
- Keep all logic local (no MQTT in this phase)

In [ ]:
# Cell purpose: Import modules and load configuration from config.yaml.
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from random import Random
from time import sleep

from simulated_city.config import load_config

cfg = load_config()
sim_cfg = cfg.simulation

seed = sim_cfg.seed if sim_cfg and sim_cfg.seed is not None else 42
step_delay_s = sim_cfg.step_delay_s if sim_cfg else 0.0
rng = Random(seed)

print(f"Loaded config. Deterministic seed={seed}, step_delay_s={step_delay_s}")

In [ ]:
# Cell purpose: Build initial in-memory person states for the Trigger agent.
@dataclass
class PersonState:
    person_id: str
    x: float
    y: float
    dx: float
    dy: float
    health_status: str


population_size = 12

# Prefer configured simulation locations if they exist; otherwise use a normalized city box.
if sim_cfg and sim_cfg.locations:
    min_x = min(loc.lon for loc in sim_cfg.locations)
    max_x = max(loc.lon for loc in sim_cfg.locations)
    min_y = min(loc.lat for loc in sim_cfg.locations)
    max_y = max(loc.lat for loc in sim_cfg.locations)
else:
    min_x, max_x = 0.0, 1.0
    min_y, max_y = 0.0, 1.0

speed_scale = 0.01 * (max_x - min_x if (max_x - min_x) > 0 else 1.0)

persons: list[PersonState] = []
for index in range(population_size):
    x = rng.uniform(min_x, max_x)
    y = rng.uniform(min_y, max_y)
    dx = rng.uniform(-speed_scale, speed_scale)
    dy = rng.uniform(-speed_scale, speed_scale)
    health_status = "infected" if index == 0 else "susceptible"
    persons.append(
        PersonState(
            person_id=f"person-{index:03d}",
            x=x,
            y=y,
            dx=dx,
            dy=dy,
            health_status=health_status,
        )
    )

infected_count = sum(1 for p in persons if p.health_status == "infected")
print(f"Initialized {len(persons)} persons. infected={infected_count}, susceptible={len(persons)-infected_count}")

In [ ]:
# Cell purpose: Run a deterministic movement loop with boundary reflection and print local state snapshots.
def _reflect(value: float, velocity: float, low: float, high: float) -> tuple[float, float]:
    next_value = value + velocity
    next_velocity = velocity

    if next_value < low:
        overflow = low - next_value
        next_value = low + overflow
        next_velocity = -next_velocity
    elif next_value > high:
        overflow = next_value - high
        next_value = high - overflow
        next_velocity = -next_velocity

    return next_value, next_velocity


total_steps = 10
start_ts = sim_cfg.start_time if sim_cfg and sim_cfg.start_time else datetime(2026, 1, 1, tzinfo=timezone.utc)
print(f"Simulation start UTC: {start_ts.isoformat()}")

for step in range(total_steps):
    moved: list[PersonState] = []
    for person in persons:
        next_x, next_dx = _reflect(person.x, person.dx, min_x, max_x)
        next_y, next_dy = _reflect(person.y, person.dy, min_y, max_y)
        moved.append(
            PersonState(
                person_id=person.person_id,
                x=next_x,
                y=next_y,
                dx=next_dx,
                dy=next_dy,
                health_status=person.health_status,
            )
        )

    persons = moved

    infected_count = sum(1 for p in persons if p.health_status == "infected")
    recovered_count = sum(1 for p in persons if p.health_status == "recovered")
    susceptible_count = len(persons) - infected_count - recovered_count

    sample = persons[0]
    print(
        f"step={step:02d} sample={sample.person_id} x={sample.x:.3f} y={sample.y:.3f} "
        f"susceptible={susceptible_count} infected={infected_count} recovered={recovered_count}"
    )

    if step_delay_s > 0:
        sleep(step_delay_s)

print("Phase 1 simulation complete (local state only, no MQTT).")

In [ ]:
# Cell purpose: Inspect a minimal publishing-ready local schema (without sending anything).
sample_person = persons[0]
preview_message = {
    "step": total_steps - 1,
    "ts": (start_ts + timedelta(minutes=15 * (total_steps - 1))).isoformat(),
    "person_id": sample_person.person_id,
    "x": round(sample_person.x, 6),
    "y": round(sample_person.y, 6),
    "health_status": sample_person.health_status,
}
print(preview_message)